# 23 - Mandatory End-to-End Evidence

Runs and persists every acceptance check listed in the README
compliance matrix. Execute this notebook only after:

1. Bronze notebooks `06`-`08` have been rerun once with no new
   files, producing zero-write audit evidence.
2. Silver, Gold, CDF, Time Travel, and optimization notebooks
   through `22` have completed successfully.

A failed test is still written to
`control.end_to_end_test_results` before the notebook raises,
making missing evidence explicit and auditable.


In [ ]:
%run ./05_bronze_common


In [ ]:
import uuid

from datetime import datetime, timezone

from delta.tables import DeltaTable
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window


dbutils.widgets.text("catalog", "abc_retail", "Target catalog")
dbutils.widgets.text("job_run_id", "manual", "Notebook or job run ID")

catalog = validate_identifier(dbutils.widgets.get("catalog").strip(), "catalog")
job_run_id = dbutils.widgets.get("job_run_id").strip()
spark.conf.set("spark.sql.session.timeZone", "UTC")

pipeline_name = "end_to_end_evidence"
pipeline_run_id = str(uuid.uuid4())
start_ts = datetime.now(timezone.utc)
evidence_table = f"{catalog}.control.end_to_end_test_results"

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS `{catalog}`.`control`.`end_to_end_test_results` (
        evidence_run_id STRING NOT NULL,
        test_name STRING NOT NULL,
        requirement STRING NOT NULL,
        status STRING NOT NULL,
        evidence_count BIGINT NOT NULL,
        violation_count BIGINT NOT NULL,
        details STRING NOT NULL,
        recorded_ts TIMESTAMP NOT NULL
    )
    USING DELTA
    TBLPROPERTIES (
        'comment' = 'Persisted pass/fail evidence for mandatory end-to-end cases'
    )
    """
)

all_sources = [
    f"{catalog}.bronze.bronze_clickstream",
    f"{catalog}.bronze.bronze_orders",
    f"{catalog}.bronze.bronze_customers",
    f"{catalog}.silver.silver_events",
    f"{catalog}.silver.silver_orders",
    f"{catalog}.silver.silver_order_status_history",
    f"{catalog}.silver.silver_customers",
    f"{catalog}.gold.gold_daily_funnel",
    f"{catalog}.gold.gold_revenue_daily",
]
upsert_audit(
    catalog, pipeline_run_id, pipeline_name, "mandatory_checks", start_ts,
    "RUNNING", all_sources, job_run_id,
)

result_schema = T.StructType(
    [
        T.StructField("evidence_run_id", T.StringType(), False),
        T.StructField("test_name", T.StringType(), False),
        T.StructField("requirement", T.StringType(), False),
        T.StructField("status", T.StringType(), False),
        T.StructField("evidence_count", T.LongType(), False),
        T.StructField("violation_count", T.LongType(), False),
        T.StructField("details", T.StringType(), False),
        T.StructField("recorded_ts", T.TimestampType(), False),
    ]
)

results = []


def add_result(test_name, requirement, evidence_count, violation_count, details):
    evidence_count = int(evidence_count)
    violation_count = int(violation_count)
    status = "PASS" if evidence_count > 0 and violation_count == 0 else "FAIL"
    results.append(
        Row(
            evidence_run_id=pipeline_run_id,
            test_name=test_name,
            requirement=requirement,
            status=status,
            evidence_count=evidence_count,
            violation_count=violation_count,
            details=details,
            recorded_ts=datetime.now(timezone.utc),
        )
    )


def latest_successful_run(pipeline):
    return (
        spark.table(f"{catalog}.control.audit_pipeline_runs")
        .filter(
            (F.col("pipeline_name") == pipeline)
            & (F.col("status") == "SUCCESS")
        )
        .orderBy(F.col("end_ts").desc())
        .first()
    )


try:
    audit = spark.table(f"{catalog}.control.audit_pipeline_runs")
    bronze_zero_runs = (
        audit.filter(
            F.col("pipeline_name").isin(
                "bronze_clickstream", "bronze_orders", "bronze_customers"
            )
            & (F.col("status") == "SUCCESS")
            & (F.col("rows_written") == 0)
        )
        .select("pipeline_name")
        .distinct()
        .count()
    )
    key_violations = (
        spark.table(f"{catalog}.silver.silver_events")
        .groupBy("event_id").count().filter(F.col("count") > 1).count()
        + spark.table(f"{catalog}.silver.silver_orders")
        .groupBy("order_id").count().filter(F.col("count") > 1).count()
        + spark.table(f"{catalog}.silver.silver_order_items")
        .groupBy("order_id", "sku").count().filter(F.col("count") > 1).count()
        + spark.table(f"{catalog}.gold.gold_daily_funnel")
        .groupBy("event_date").count().filter(F.col("count") > 1).count()
        + spark.table(f"{catalog}.gold.gold_revenue_daily")
        .groupBy("revenue_date", "country").count().filter(F.col("count") > 1).count()
        + spark.table(f"{catalog}.gold.gold_customer_360")
        .groupBy("user_id").count().filter(F.col("count") > 1).count()
        + spark.table(f"{catalog}.gold.gold_data_quality_dashboard")
        .groupBy("quality_date", "source_name", "quality_status", "error_type")
        .count().filter(F.col("count") > 1).count()
    )
    add_result(
        "same_file_rerun",
        "Repeated pipeline runs do not duplicate Bronze, Silver, or Gold",
        bronze_zero_runs,
        key_violations + (0 if bronze_zero_runs == 3 else 1),
        "Requires successful zero-write reruns for all three Bronze pipelines and unique downstream business keys.",
    )

    bronze_clickstream = (
        spark.table(f"{catalog}.bronze.bronze_clickstream")
        .withColumn("event_id_parsed", F.get_json_object("raw_payload", "$.event_id"))
        .withColumn(
            "source_ingest_time_parsed",
            F.expr("try_to_timestamp(get_json_object(raw_payload, '$.ingest_time'))"),
        )
    )
    duplicate_ids = (
        bronze_clickstream.filter(F.col("event_id_parsed").isNotNull())
        .groupBy("event_id_parsed")
        .count()
        .filter(F.col("count") > 1)
        .select("event_id_parsed")
    )
    duplicate_rank = Window.partitionBy("event_id_parsed").orderBy(
        F.col("source_ingest_time_parsed").desc_nulls_last(),
        F.col("ingest_ts").desc(),
        F.col("source_file").desc(),
    )
    expected_duplicate_winners = (
        bronze_clickstream.join(duplicate_ids, "event_id_parsed", "inner")
        .withColumn("_rank", F.row_number().over(duplicate_rank))
        .filter(F.col("_rank") == 1)
        .select(
            F.col("event_id_parsed").alias("event_id"),
            F.col("source_file").alias("expected_source_file"),
        )
    )
    duplicate_winner_violations = (
        expected_duplicate_winners.join(
            spark.table(f"{catalog}.silver.silver_events")
            .select("event_id", "source_file"),
            "event_id",
            "left",
        )
        .filter(
            F.col("source_file").isNull()
            | (F.col("source_file") != F.col("expected_source_file"))
        )
        .count()
    )
    add_result(
        "business_duplicate_event",
        "Bronze preserves retries and Silver keeps the newest event_id record",
        duplicate_ids.count(),
        duplicate_winner_violations,
        "Compares each duplicated Bronze event_id winner with the selected Silver source_file.",
    )

    parsed_event_time = F.expr(
        "try_to_timestamp(get_json_object(raw_payload, '$.event_time'))"
    )
    late_events = (
        bronze_clickstream.withColumn("event_time_parsed", parsed_event_time)
        .filter(
            F.col("event_id_parsed").isNotNull()
            & F.col("event_time_parsed").isNotNull()
            & (F.datediff("load_date", F.to_date("event_time_parsed")) >= 7)
        )
        .select(
            F.col("event_id_parsed").alias("event_id"),
            F.to_date("event_time_parsed").alias("event_date"),
        )
        .dropDuplicates()
    )
    late_event_violations = (
        late_events.join(
            spark.table(f"{catalog}.silver.silver_events")
            .select("event_id", "event_date"),
            ["event_id", "event_date"],
            "left_anti",
        ).count()
        + late_events.select("event_date").distinct().join(
            spark.table(f"{catalog}.gold.gold_daily_funnel").select("event_date"),
            "event_date",
            "left_anti",
        ).count()
    )
    cdf_success = (
        audit.filter(
            (F.col("pipeline_name") == "gold_daily_funnel_cdf")
            & (F.col("status") == "SUCCESS")
            & (F.col("rows_written") > 0)
        ).count()
    )
    add_result(
        "seven_day_late_event",
        "Late event is conformed and its historical funnel date is present",
        late_events.count(),
        late_event_violations + (0 if cdf_success else 1),
        "Validates late Bronze events against Silver/event_date, Gold funnel, and a successful CDF update.",
    )

    device_raw = bronze_clickstream.withColumn(
        "device_raw",
        F.coalesce(
            F.get_json_object("raw_payload", "$.device"),
            F.get_json_object("_rescued_data", "$.device"),
        ),
    )
    struct_devices = device_raw.filter(F.col("device_raw").startswith("{")).count()
    string_devices = device_raw.filter(
        F.col("device_raw").isNotNull()
        & ~F.col("device_raw").startswith("{")
    ).count()
    schema_mismatch = (
        spark.table(f"{catalog}.quarantine.clickstream_errors")
        .filter(F.col("error_code") == "schema_mismatch")
        .count()
    )
    normalized_device_violations = (
        spark.table(f"{catalog}.silver.silver_events")
        .filter(F.col("device.os").isNull() | F.col("device.model").isNull())
        .count()
    )
    add_result(
        "device_schema_drift",
        "Struct and string devices normalize; incompatible shapes quarantine",
        min(struct_devices, string_devices, schema_mismatch),
        normalized_device_violations,
        f"struct_inputs={struct_devices}, string_inputs={string_devices}, quarantined_mismatches={schema_mismatch}",
    )

    orders = spark.table(f"{catalog}.silver.silver_orders")
    order_history = spark.table(f"{catalog}.silver.silver_order_status_history")
    multi_update_orders = (
        order_history.groupBy("order_id").count().filter(F.col("count") > 1).count()
    )
    stale_order_violations = (
        orders.alias("orders")
        .join(
            order_history.groupBy("order_id").agg(
                F.max("status_ts").alias("max_status_ts")
            ).alias("history"),
            "order_id",
        )
        .filter(F.col("orders.updated_at") != F.col("history.max_status_ts"))
        .count()
    )
    add_result(
        "stale_order_update",
        "Older order updates cannot overwrite the newest SCD1 state",
        multi_update_orders,
        stale_order_violations,
        "Compares every latest Silver order updated_at with max status history timestamp.",
    )

    revenue_history = (
        order_history.filter(F.col("status").isin("paid", "cancelled", "returned"))
        .withColumn("revenue_date", F.to_date("status_ts"))
        .withColumn("country", F.coalesce("country", F.lit("UNKNOWN")))
    )
    expected_revenue = revenue_history.groupBy("revenue_date", "country").agg(
        F.sum(
            F.when(F.col("status") == "paid", F.col("subtotal_usd"))
            .when(
                F.col("status").isin("cancelled", "returned"),
                -F.col("subtotal_usd"),
            )
            .otherwise(F.lit(0))
        ).cast("decimal(28,2)").alias("expected_net")
    )
    revenue_violations = (
        expected_revenue.alias("expected")
        .join(
            spark.table(f"{catalog}.gold.gold_revenue_daily").alias("actual"),
            ["revenue_date", "country"],
            "full",
        )
        .filter(
            F.coalesce(
                F.abs(
                    F.col("expected.expected_net")
                    - F.col("actual.net_revenue_usd")
                ),
                F.lit(1),
            )
            > F.lit(0.01)
        )
        .count()
    )
    reversal_count = revenue_history.filter(
        F.col("status").isin("cancelled", "returned")
    ).count()
    add_result(
        "order_status_timeline",
        "Paid adds and cancelled/returned subtract net revenue",
        reversal_count,
        revenue_violations,
        "Reconciles Gold net revenue to signed Silver order status history.",
    )

    fx_errors = (
        spark.table(f"{catalog}.quarantine.orders_errors")
        .filter(F.col("error_code") == "fx_rate_missing")
        .count()
    )
    missing_fx_silver = orders.filter(F.col("currency") == "GBP").count()
    add_result(
        "missing_fx_rate",
        "Missing FX records quarantine and do not enter USD Silver/Gold",
        fx_errors,
        missing_fx_silver,
        "Generated GBP rows have no reference FX rate and must be excluded from Silver.",
    )

    bronze_customers = (
        spark.table(f"{catalog}.bronze.bronze_customers")
        .withColumn(
            "updated_at_parsed",
            F.expr("try_to_timestamp(get_json_object(raw_payload, '$.updated_at'))"),
        )
    )
    late_customer_rows = bronze_customers.filter(
        F.col("updated_at_parsed").isNotNull()
        & (F.datediff("load_date", F.to_date("updated_at_parsed")) >= 1)
    ).count()
    customers = spark.table(f"{catalog}.silver.silver_customers")
    customer_timeline = Window.partitionBy("user_id").orderBy("effective_from")
    customer_interval_violations = (
        customers.withColumn(
            "_previous_to", F.lag("effective_to").over(customer_timeline)
        )
        .filter(
            F.col("_previous_to").isNotNull()
            & (F.col("effective_from") <= F.col("_previous_to"))
        )
        .count()
        + customers.filter("is_current")
        .groupBy("user_id").count().filter(F.col("count") != 1).count()
    )
    add_result(
        "late_customer_update",
        "Late customer updates form a valid non-overlapping SCD2 timeline",
        late_customer_rows,
        customer_interval_violations,
        "Checks late Bronze customer versions, one current row, and no SCD2 overlap.",
    )

    customer_contact_errors = (
        spark.table(f"{catalog}.quarantine.customers_errors")
        .filter(
            (F.col("error_stage") == "silver")
            & (F.col("error_code") == "schema_mismatch")
        )
        .count()
    )
    invalid_current_contacts = customers.filter(
        F.col("is_current")
        & (
            F.col("phone").isNull()
            | (F.trim("phone") == "")
            | (
                F.col("email").isNotNull()
                & (
                    (F.trim("email") == "")
                    | ~F.col("email").rlike(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")
                )
            )
        )
    ).count()
    add_result(
        "missing_invalid_customer_contact",
        "Missing or invalid customer contact follows explicit quarantine rule",
        customer_contact_errors,
        invalid_current_contacts,
        "Customer Silver classifies missing phone or invalid email as schema_mismatch; current phone must be non-null.",
    )

    time_travel_evidence = (
        spark.table(f"{catalog}.control.time_travel_evidence")
        .select("table_name")
        .distinct()
        .count()
        if spark.catalog.tableExists(f"{catalog}.control.time_travel_evidence")
        else 0
    )
    add_result(
        "time_travel",
        "Previous Silver and Gold versions remain queryable",
        time_travel_evidence,
        0 if time_travel_evidence >= 2 else 1,
        "Requires persisted previous-version evidence for at least one Silver and one Gold table.",
    )

    watermark_exists = (
        spark.table(f"{catalog}.control.pipeline_watermarks")
        .filter(
            (F.col("pipeline_name") == "gold_daily_funnel_cdf")
            & (F.col("watermark_name") == "silver_events_cdf_version")
        )
        .count()
    )
    add_result(
        "cdf_incremental_update",
        "Silver CDF incrementally updates a Gold table",
        cdf_success,
        0 if cdf_success and watermark_exists else 1,
        "Requires a successful positive-write CDF audit run and persisted Delta version watermark.",
    )

    optimization_metrics = (
        spark.table(f"{catalog}.control.delta_optimization_metrics")
        if spark.catalog.tableExists(f"{catalog}.control.delta_optimization_metrics")
        else None
    )
    optimized_tables = (
        optimization_metrics.groupBy("table_name")
        .agg(F.countDistinct("metric_stage").alias("stages"))
        .filter(F.col("stages") >= 2)
        .count()
        if optimization_metrics is not None
        else 0
    )
    add_result(
        "optimization",
        "Before/after file metrics exist for at least two optimized tables",
        optimized_tables,
        0 if optimized_tables >= 2 else 1,
        "Requires before/after numFiles, sizeInBytes, and average file-size evidence.",
    )

    required_pipelines = {
        "load_dim_fx_rates",
        "bronze_clickstream",
        "bronze_orders",
        "bronze_customers",
        "silver_events",
        "silver_orders",
        "silver_order_items",
        "silver_customers_scd2",
        "gold_daily_funnel",
        "gold_revenue_daily",
        "gold_customer_360",
        "gold_data_quality_dashboard",
        "gold_daily_funnel_cdf",
        "time_travel_demo",
        "delta_optimization",
    }
    required_audit = audit.filter(F.col("pipeline_name").isin(sorted(required_pipelines)))
    successful_pipelines = {
        row.pipeline_name
        for row in required_audit.filter(F.col("status") == "SUCCESS")
        .select("pipeline_name")
        .distinct()
        .collect()
    }
    missing_pipelines = sorted(required_pipelines - successful_pipelines)
    incomplete_audits = required_audit.filter(
        F.col("end_ts").isNull()
        | F.col("source_paths").isNull()
        | (
            (F.col("status") == "SUCCESS")
            & (
                F.col("rows_read").isNull()
                | F.col("rows_written").isNull()
                | F.col("rows_quarantined").isNull()
            )
        )
    ).count()
    add_result(
        "audit_completeness",
        "Every operational pipeline has final audit status and required metrics",
        len(required_pipelines) - len(missing_pipelines),
        incomplete_audits + len(missing_pipelines),
        f"missing_successful_pipelines={missing_pipelines}",
    )

    result_df = spark.createDataFrame(results, result_schema)
    result_df.write.format("delta").mode("append").saveAsTable(evidence_table)
    result_df.orderBy("test_name").display()

    failed = result_df.filter(F.col("status") != "PASS").count()
    upsert_audit(
        catalog, pipeline_run_id, pipeline_name, "mandatory_checks", start_ts,
        "SUCCESS" if failed == 0 else "FAILED", all_sources, job_run_id,
        end_ts=datetime.now(timezone.utc), rows_read=0,
        rows_written=len(results), rows_quarantined=failed,
        error_message=None if failed == 0 else f"{failed} evidence checks failed",
    )
    if failed:
        raise RuntimeError(f"{failed} mandatory end-to-end evidence checks failed")
except Exception as exc:
    existing = (
        spark.table(evidence_table)
        .filter(F.col("evidence_run_id") == pipeline_run_id)
        .count()
    )
    if not existing:
        upsert_audit(
            catalog, pipeline_run_id, pipeline_name, "mandatory_checks", start_ts,
            "FAILED", all_sources, job_run_id,
            end_ts=datetime.now(timezone.utc), error_message=str(exc)[:4000],
        )
    raise
